In [6]:
from sagemaker.experiments import Run
from sagemaker.pytorch import PyTorch
from sagemaker import Session
from sagemaker import get_execution_role
from sagemaker.utils import unique_name_from_base
from sagemaker.debugger import TensorBoardOutputConfig
import os

# Notes on running on AWS SM:
# instance type: numGPUs, memPerGPU, max images per batch (this is a per GPU number), approximate time per epoch, cost, cost multiplier rel g4dn..
# "ml.g4dn.4xlarge": 1, 16, 20, 3.32 min per epoch, $1.806 per hr., x1
# "ml.p3.8xlarge": 4, 16, 20, 1.41 min per epoch, $14.688, x8.132
# "ml.p3.16xlarge": 8, 16, 20, 1min per epoch, $28.152, x15.588
# "ml.p4de.24xlarge": 8, 80, 20

# If running notebook inside a SageMaker Notebook or Studio instance,
# can retrieve the execution role associated with the instance
role = "arn:aws:iam::159929462505:role/LLNL_User_Roles_Sagemaker" # get_execution_role()

metric_def = [
    {"Name": "train:loss", "Regex": "train_loss=(.*?)[,\]]"},
    {"Name": "val:loss", "Regex": "val_loss=(.*?)[,\]]"},
    {"Name": "learning_rate", "Regex": "lr=(.*?)[,\]]"},
    {"Name": "val:acc", "Regex": "val_acc=(.*?)[,\]]"},
    {"Name": "val:jaccard", "Regex": "val_jaccard=(.*?)[,\]]"},
    {"Name": "val:dice", "Regex": "val_dice=(.*?)[,\]]"},
    {"Name": "test:loss", "Regex": "test_loss=(.*?)[,\]]"},
    {"Name": "test:acc", "Regex": "test_acc=(.*?)[,\]]"},
    {"Name": "test:jaccard", "Regex": "test_jaccard=(.*?)[,\]]"},
    {"Name": "test:dice", "Regex": "test_dice=(.*?)[,\]]"},
]


train_image = "s3://image-segmentation-2k/images"
train_mask = "s3://image-segmentation-2k/masks"

experiment_name = "tf-test-unet-mz"
project_name = experiment_name
run_name = unique_name_from_base(experiment_name)
print( run_name )

# If launching multiple jobs within a short period of time utilizing the same instance types
# Update this value to a time in seconds to create a Warm Pool. This will enable faster 
# launching of subsequent training jobs after the first. NOTE: Costs are associated with keeping a warm pool alive
# More details here: https://docs.aws.amazon.com/sagemaker/latest/dg/train-warm-pools.html
warm_pool_duration = 10


###tensorboard_output_config = TensorBoardOutputConfig(
###    s3_output_path=os.path.join(f"s3://{bucket}", 'tensorboard_logs', job_id, "tensorboard"),
###    container_local_output_path="/opt/ml/output/tensorboard"
###)

hyperparameters = {
    "num_workers": 8,
    "model_name": "UNet",
    "model_params": '''{"param": "value"}''',
    # Note, with Pytorch Lightning, this is a per GPU batch count. PL will create a batch per device
    # behind the scenes. No need to adjust if changing instances with different gpu counts. 
    # Recommend adjust if changes instances that have different GPU memory (e.g., p3.* -> p3dn.* -> p4d.* -> p4de.*)
    "batch_size": 20,
    "epochs": 5,
    "learning_rate": 1e-4,
    "lr_schedular": "cosine_warmup",
    "image_mode": "grayscale",
    # Added a new helper function called str2bool in train.py to aid in this problem.
    # Can now use "False", "No" or 0 to indicate a negative boolean
    # and "True", "Yes" or 1 to indicate a positive boolean.
    "use_zero":"True", # Note comment on bool as type. Empty String => "" = False; Any string => "Hello" | "True" | "Flase" = True; https://docs.python.org/3/library/argparse.html#type
    "use_amp": "True", # See above comment
    # Data Augmentation Parameters
    "image_size": 512,
    "use_random_resize": "True",
    "use_random_rotation": "False",
    "color_jitter": 0.1,
    "center_crop": "True",
    "center_crop_size": 1200,
    "test_image_size": 1200,
    "test_batch_size": 1,
    "run_name": run_name,
    "project_name": project_name,
}

#with Run(experiment_name=experiment_name, run_name=run_name, sagemaker_session=Session()) as run:
estimator = PyTorch(
    entry_point="train.py",
    # Point to the repo root one level up,
    # this notebook is at {repo}/notebooks/LaunchTrainingJob.ipynb
    source_dir="../src/",  
    role=role,
    framework_version="2.0.1",
    py_version="py310",
    instance_count=1,
    instance_type= 'ml.g4dn.4xlarge', #'ml.p3.8xlarge', #'ml.p3.16xlarge', #'ml.p4de.24xlarge', # 80GB A100 GPU   "ml.g4dn.4xlarge", 
    hyperparameters=hyperparameters, 
    # Default input_mode is "File" mode which will download a copy of all contents in S3
    # locally prior to training. "FastFile" will instead stream files from S3 as they are
    # requested. This essentially makes S3 a mounted drive to stream the data from.
    # More information can be found here: 
    # https://docs.aws.amazon.com/sagemaker/latest/dg/model-access-training-data.html
    #input_mode="File",
    metric_definitions=metric_def,
    # NOTE: If launching job from outside SageMaker Domain
    # tensorboard_ouptut_config may cause errors when running the training job. Investigating cause and remediation.
    # If job fails with "internal error", comment out below line and relaunch job.    
    environment={"WANDB_API_KEY": "33ae28acc6b5063bcc215f3928bcf533fd87107d"},
    #tensorboard_output_config=tensorboard_output_config,
    keep_alive_period_in_seconds=warm_pool_duration,
)

estimator.fit(
    {"train_image": train_image, "train_mask": train_mask}
)

KeyboardInterrupt: 